# ENG5338 CW2 — Virtual CO sensor (UCI Air Quality)

## Part A — M1: data, question, constraints, shared protocol and linear reference
Sections below (data → features → split/CV → scorers → harness → LR/Ridge) are the **frozen shared protocol**. Do not edit without telling M1.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
from pathlib import Path
# Works from the repo root, from notebooks/, or in Colab after cloning the repo
DATA_PATH = next(p for p in [Path("data/air-quality.csv"), Path("../data/air-quality.csv"),
                             Path("eng5338-cw2-air-quality/data/air-quality.csv")] if p.exists())
df = pd.read_csv(DATA_PATH, parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
print(df.shape)
print(df["datetime"].min(), "->", df["datetime"].max())
df.head()

StopIteration: 

In [ ]:
print("NaN before:", df.isna().sum().sum())
df = df.replace(-200, np.nan)

missing = df.isna().sum().to_frame("missing")
missing["pct"] = (missing["missing"] / len(df) * 100).round(1)
missing.sort_values("pct", ascending=False)

In [ ]:
df = df.drop(columns=["nmhc_gt"])
df = df.dropna(subset=["co_gt"]).reset_index(drop=True)
print(len(df), "rows remain")

In [ ]:
SENSOR_COLS  = ["pt08_s1_co", "pt08_s2_nmhc", "pt08_s3_nox", "pt08_s4_no2", "pt08_s5_o3"]
WEATHER_COLS = ["temperature_c", "relative_humidity_pct", "absolute_humidity"]
TARGET = "co_gt"

df[[TARGET] + SENSOR_COLS + WEATHER_COLS].describe().T.round(2)

In [ ]:
df["hour"] = df["datetime"].dt.hour

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET], bins=50, ax=ax[0])
ax[0].set_title("Distribution of true CO (mg/m³)")

df.groupby("hour")[TARGET].mean().plot(ax=ax[1], marker="o")
ax[1].set_title("Mean CO by hour of day")
ax[1].set_ylabel("CO (mg/m³)")
plt.tight_layout(); plt.show()

In [ ]:
corr = df[[TARGET] + SENSOR_COLS + WEATHER_COLS].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.tight_layout(); plt.show()

corr[TARGET].drop(TARGET).sort_values(ascending=False)

In [ ]:
daily = df.set_index("datetime")[[TARGET, "pt08_s1_co"]].resample("D").mean()

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(daily.index, daily[TARGET], label="True CO (co_gt)", color="tab:blue")
ax1.set_ylabel("True CO (mg/m³)")
ax2 = ax1.twinx()
ax2.plot(daily.index, daily["pt08_s1_co"], label="CO sensor (pt08_s1_co)", color="tab:orange", alpha=0.7)
ax2.set_ylabel("Sensor response")
ax1.set_title("Daily mean: true CO vs low-cost CO sensor over time")
fig.legend(loc="upper left", bbox_to_anchor=(0.07, 0.9))
plt.tight_layout(); plt.show()

In [ ]:
df["hour"] = df["datetime"].dt.hour            # 0 to 23
df["weekday"] = df["datetime"].dt.weekday      # 0 = Monday ... 6 = Sunday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)   # 1 if Sat/Sun, else 0

In [ ]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

In [ ]:
SENSOR_COLS  = ["pt08_s1_co", "pt08_s2_nmhc", "pt08_s3_nox", "pt08_s4_no2", "pt08_s5_o3"]
WEATHER_COLS = ["temperature_c", "relative_humidity_pct", "absolute_humidity"]
TIME_COLS    = ["hour_sin", "hour_cos", "weekday", "is_weekend"]

FEATURES = SENSOR_COLS + WEATHER_COLS + TIME_COLS
TARGET = "co_gt"

print(len(FEATURES), "features:", FEATURES)

In [ ]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]

print(f"Train: {len(train_df)} rows  {train_df.datetime.min():%Y-%m-%d} -> {train_df.datetime.max():%Y-%m-%d}")
print(f"Test : {len(test_df)} rows  {test_df.datetime.min():%Y-%m-%d} -> {test_df.datetime.max():%Y-%m-%d}")

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

CV = TimeSeriesSplit(n_splits=5)
for i, (tr, va) in enumerate(CV.split(X_train)):
    print(f"Fold {i+1}: train {len(tr):5d} rows | validate {len(va):4d} rows "
          f"({train_df.datetime.iloc[va[0]]:%Y-%m-%d} -> {train_df.datetime.iloc[va[-1]]:%Y-%m-%d})")

In [ ]:
from sklearn.metrics import make_scorer

TOLERANCE = 0.5          # mg/m³ – constraint 1
TARGET_PCT = 90          # %     – constraint 1
LATENCY_LIMIT_MS = 1.0   # ms per sample – constraint 2

def pct_within(y_true, y_pred, tol=TOLERANCE):
    return 100 * np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)) <= tol)

SCORING = {
    "mae":    "neg_mean_absolute_error",
    "rmse":   "neg_root_mean_squared_error",
    "r2":     "r2",
    "within": make_scorer(pct_within),
}
PRIMARY = "mae"

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def make_preprocessor(scale=True):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    return ColumnTransformer([("num", Pipeline(steps), FEATURES)])

In [ ]:
from sklearn.model_selection import GridSearchCV
import platform, os

RUNTIME = f"{platform.processor() or platform.machine()} | {os.cpu_count()} CPU cores | Python {platform.python_version()}"
print("Runtime:", RUNTIME)

results = []   # every model adds one row here

def evaluate_search(name, gs):
    r, i = gs.cv_results_, gs.best_index_
    row = {
        "model": name,
        "train_MAE": -r["mean_train_mae"][i],
        "val_MAE":   -r["mean_test_mae"][i],
        "val_MAE_std": r["std_test_mae"][i],
        "train_R2": r["mean_train_r2"][i],
        "val_R2":   r["mean_test_r2"][i],
        "train_within_%": r["mean_train_within"][i],
        "val_within_%":   r["mean_test_within"][i],
        "mean_fit_time_s": r["mean_fit_time"][i],
        "best_params": gs.best_params_,
    }
    row["overfit_gap_MAE"] = row["val_MAE"] - row["train_MAE"]
    results.append(row)
    return pd.Series(row)

def run_search(name, pipe, grid):
    gs = GridSearchCV(pipe, grid, cv=CV, scoring=SCORING, refit=PRIMARY,
                      return_train_score=True, n_jobs=-1)
    gs.fit(X_train, y_train)
    return gs, evaluate_search(name, gs)

In [ ]:
from sklearn.linear_model import LinearRegression
lin_pipe = Pipeline([("prep", make_preprocessor()), ("model", LinearRegression())])
lin_gs, lin_res = run_search("Linear Regression", lin_pipe, {})
lin_res


In [ ]:
from sklearn.linear_model import Ridge
ridge_pipe = Pipeline([("prep", make_preprocessor()), ("model", Ridge())])
ridge_grid = {"model__alpha": np.logspace(-3, 3, 13)}
ridge_gs, ridge_res = run_search("Ridge (baseline)", ridge_pipe, ridge_grid)
ridge_res

## Part B — M2: Random Forest


In [ ]:
!cat /content/eng5338-cw2-air-quality/README.md

# ENG5338 CW2 — Virtual CO Sensor (UCI Air Quality)

Group notebook: `ENG5338_CW2_notebook.ipynb`

| Part | Owner | Content |
|------|-------|---------|
| A | M1 | Data cleaning, EDA, question & constraints, shared protocol (chronological 80/20 split, `TimeSeriesSplit(5)`, scorers, `run_search`), Linear/Ridge reference |
| B | M2 | Random Forest |
| C | M3 | HistGradientBoosting |
| D | M4 | SVR / KNN |
| E | M1 | Overfitting analysis |
| F | M5 | Computational cost |
| G | M5 | Final recommended pipeline, scored once on the test set |

**To run:** put `air-quality.csv` in the same folder as the notebook, then run all cells. It needs numpy, pandas, scikit-learn, matplotlib and seaborn.


In [ ]:

import os
import requests
import zipfile
import io
import pandas as pd

# Download the official UCI Air Quality dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip"

response = requests.get(url)
response.raise_for_status()

# Read the Excel dataset from the ZIP file
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("AirQualityUCI.xlsx") as f:
        raw = pd.read_excel(f, skipfooter=2)

# Remove completely empty rows
raw = raw.dropna(how="all")

# Convert date and time correctly
raw["datetime"] = pd.to_datetime(
    raw["Date"].astype(str) + " " + raw["Time"].astype(str),
    dayfirst=True
)

# Rename columns to match the coursework notebook
raw = raw.rename(columns={
    "CO(GT)": "co_gt",
    "PT08.S1(CO)": "pt08_s1_co",
    "NMHC(GT)": "nmhc_gt",
    "C6H6(GT)": "c6h6_gt",
    "PT08.S2(NMHC)": "pt08_s2_nmhc",
    "NOx(GT)": "nox_gt",
    "PT08.S3(NOx)": "pt08_s3_nox",
    "NO2(GT)": "no2_gt",
    "PT08.S4(NO2)": "pt08_s4_no2",
    "PT08.S5(O3)": "pt08_s5_o3",
    "T": "temperature_c",
    "RH": "relative_humidity_pct",
    "AH": "absolute_humidity"
})

# Arrange columns in the coursework order
columns = [
    "datetime", "co_gt", "pt08_s1_co", "nmhc_gt",
    "c6h6_gt", "pt08_s2_nmhc", "nox_gt", "pt08_s3_nox",
    "no2_gt", "pt08_s4_no2", "pt08_s5_o3",
    "temperature_c", "relative_humidity_pct",
    "absolute_humidity"
]

df = raw[columns].copy()

# Save dataset in the expected location
os.makedirs("data", exist_ok=True)
df.to_csv("data/air-quality.csv", index=False)

# Display confirmation
print("Dataset restored successfully!")
print("Dataset shape:", df.shape)
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())
print("\nFirst three rows:")
print(df.head(3))


ValueError: time data "2004-03-13 00:00:00" doesn't match format "%Y-%d-%m %H:%M:%S", at position 54. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
print(raw[["Date", "Time"]].head(10).to_string())
print("\nData types:")
print(raw[["Date", "Time"]].dtypes)

        Date      Time
0 2004-03-10  18:00:00
1 2004-03-10  19:00:00
2 2004-03-10  20:00:00
3 2004-03-10  21:00:00
4 2004-03-10  22:00:00
5 2004-03-10  23:00:00
6 2004-03-11  00:00:00
7 2004-03-11  01:00:00
8 2004-03-11  02:00:00
9 2004-03-11  03:00:00

Data types:
Date    datetime64[ns]
Time            object
dtype: object


In [ ]:

import os
import requests
import zipfile
import io
import pandas as pd

# --------------------------------------------------
# 1. Download the official UCI Air Quality dataset
# --------------------------------------------------

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip"

response = requests.get(url, timeout=60)
response.raise_for_status()

print("Dataset downloaded successfully.")

# --------------------------------------------------
# 2. Read the Excel file
# --------------------------------------------------

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("AirQualityUCI.xlsx") as f:
        raw = pd.read_excel(f, skipfooter=2, engine="openpyxl")

# Remove completely empty rows
raw = raw.dropna(how="all").reset_index(drop=True)

print("Original dataset loaded:", raw.shape)

# --------------------------------------------------
# 3. Combine Date and Time correctly
# Date is already datetime64; Time is HH:MM:SS
# --------------------------------------------------

raw["datetime"] = (
    raw["Date"].dt.normalize()
    + pd.to_timedelta(raw["Time"].astype(str), errors="coerce")
)

# Remove rows with invalid dates or times
raw = raw.dropna(subset=["datetime"]).copy()

# --------------------------------------------------
# 4. Rename columns to match the coursework
# --------------------------------------------------

raw = raw.rename(columns={
    "CO(GT)": "co_gt",
    "PT08.S1(CO)": "pt08_s1_co",
    "NMHC(GT)": "nmhc_gt",
    "C6H6(GT)": "c6h6_gt",
    "PT08.S2(NMHC)": "pt08_s2_nmhc",
    "NOx(GT)": "nox_gt",
    "PT08.S3(NOx)": "pt08_s3_nox",
    "NO2(GT)": "no2_gt",
    "PT08.S4(NO2)": "pt08_s4_no2",
    "PT08.S5(O3)": "pt08_s5_o3",
    "T": "temperature_c",
    "RH": "relative_humidity_pct",
    "AH": "absolute_humidity"
})

# --------------------------------------------------
# 5. Select the original coursework columns
# --------------------------------------------------

columns = [
    "datetime",
    "co_gt",
    "pt08_s1_co",
    "nmhc_gt",
    "c6h6_gt",
    "pt08_s2_nmhc",
    "nox_gt",
    "pt08_s3_nox",
    "no2_gt",
    "pt08_s4_no2",
    "pt08_s5_o3",
    "temperature_c",
    "relative_humidity_pct",
    "absolute_humidity"
]

df = raw[columns].copy()

# Sort chronologically
df = df.sort_values("datetime").reset_index(drop=True)

# --------------------------------------------------
# 6. Save the dataset in the expected location
# --------------------------------------------------

os.makedirs("data", exist_ok=True)

df.to_csv("data/air-quality.csv", index=False)

# --------------------------------------------------
# 7. Verify the restored dataset
# --------------------------------------------------

print("\nDataset restored successfully!")
print("Dataset shape:", df.shape)
print("First date:", df["datetime"].min())
print("Last date:", df["datetime"].max())
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst three rows:")
display(df.head(3))

print("\nSaved file exists:", os.path.exists("data/air-quality.csv"))


Dataset downloaded successfully.
Original dataset loaded: (9355, 15)

Dataset restored successfully!
Dataset shape: (9355, 14)
First date: 2004-03-10 18:00:00
Last date: 2005-04-04 12:00:00

Column names:
['datetime', 'co_gt', 'pt08_s1_co', 'nmhc_gt', 'c6h6_gt', 'pt08_s2_nmhc', 'nox_gt', 'pt08_s3_nox', 'no2_gt', 'pt08_s4_no2', 'pt08_s5_o3', 'temperature_c', 'relative_humidity_pct', 'absolute_humidity']

First three rows:


,datetime,co_gt,pt08_s1_co,nmhc_gt,c6h6_gt,pt08_s2_nmhc,nox_gt,pt08_s3_nox,no2_gt,pt08_s4_no2,pt08_s5_o3,temperature_c,relative_humidity_pct,absolute_humidity
0,2004-03-10 18:00:00,2.6,1360.00,150,11.881723,1045.50,166.0,1056.25,113.0,1692.00,1267.50,13.6,48.875001,0.757754
1,2004-03-10 19:00:00,2.0,1292.25,112,9.397165,954.75,103.0,1173.75,92.0,1558.75,972.25,13.3,47.700000,0.725487
2,2004-03-10 20:00:00,2.2,1402.00,88,8.997817,939.25,131.0,1140.00,114.0,1554.50,1074.00,11.9,53.975000,0.750239



Saved file exists: True


In [ ]:
print("Original Excel data:")
print(raw[["Date", "Time"]].tail(8).to_string(index=False))

print("\nRestored dataset:")
print(df[["datetime", "co_gt"]].tail(8).to_string(index=False))

print("\nShapes:", raw.shape, df.shape)

Original Excel data:
      Date     Time
2005-04-04 05:00:00
2005-04-04 06:00:00
2005-04-04 07:00:00
2005-04-04 08:00:00
2005-04-04 09:00:00
2005-04-04 10:00:00
2005-04-04 11:00:00
2005-04-04 12:00:00

Restored dataset:
           datetime  co_gt
2005-04-04 05:00:00    0.5
2005-04-04 06:00:00    1.1
2005-04-04 07:00:00    4.0
2005-04-04 08:00:00    5.0
2005-04-04 09:00:00    3.9
2005-04-04 10:00:00    3.1
2005-04-04 11:00:00    2.4
2005-04-04 12:00:00    2.4

Shapes: (9355, 16) (9355, 14)


In [ ]:

import os
import io
import zipfile
import requests
import pandas as pd

# Download the official UCI dataset archive
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip"

response = requests.get(url, timeout=60)
response.raise_for_status()

# Read the original CSV, not the Excel version
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    csv_file = next(
        name for name in z.namelist()
        if name.lower().endswith(".csv")
    )

    with z.open(csv_file) as f:
        raw = pd.read_csv(
            f,
            sep=";",
            decimal=",",
            encoding="latin1"
        )

# Remove completely empty rows and columns
raw = raw.dropna(axis=0, how="all")
raw = raw.dropna(axis=1, how="all")

# Handle the original date/time columns
if "Date_Time" in raw.columns:
    raw["datetime"] = pd.to_datetime(
        raw["Date_Time"],
        format="%d/%m/%Y %H.%M.%S"
    )
else:
    raw["datetime"] = pd.to_datetime(
        raw["Date"].astype(str) + " " + raw["Time"].astype(str),
        format="%d/%m/%Y %H.%M.%S"
    )

# Rename columns to match the coursework notebook
raw = raw.rename(columns={
    "CO(GT)": "co_gt",
    "PT08.S1(CO)": "pt08_s1_co",
    "NMHC(GT)": "nmhc_gt",
    "C6H6(GT)": "c6h6_gt",
    "PT08.S2(NMHC)": "pt08_s2_nmhc",
    "NOx(GT)": "nox_gt",
    "PT08.S3(NOx)": "pt08_s3_nox",
    "NO2(GT)": "no2_gt",
    "PT08.S4(NO2)": "pt08_s4_no2",
    "PT08.S5(O3)": "pt08_s5_o3",
    "T": "temperature_c",
    "RH": "relative_humidity_pct",
    "AH": "absolute_humidity"
})

# Select the 14 coursework columns
columns = [
    "datetime", "co_gt", "pt08_s1_co", "nmhc_gt",
    "c6h6_gt", "pt08_s2_nmhc", "nox_gt", "pt08_s3_nox",
    "no2_gt", "pt08_s4_no2", "pt08_s5_o3",
    "temperature_c", "relative_humidity_pct",
    "absolute_humidity"
]

df = raw[columns].copy()
df = df.sort_values("datetime").reset_index(drop=True)

# Save the dataset
os.makedirs("data", exist_ok=True)
df.to_csv("data/air-quality.csv", index=False)

# Verify
print("Dataset restored!")
print("Shape:", df.shape)
print("First timestamp:", df["datetime"].min())
print("Last timestamp:", df["datetime"].max())
print("\nFirst three rows:")
display(df.head(3))


Dataset restored!
Shape: (9357, 14)
First timestamp: 2004-03-10 18:00:00
Last timestamp: 2005-04-04 14:00:00

First three rows:


,datetime,co_gt,pt08_s1_co,nmhc_gt,c6h6_gt,pt08_s2_nmhc,nox_gt,pt08_s3_nox,no2_gt,pt08_s4_no2,pt08_s5_o3,temperature_c,relative_humidity_pct,absolute_humidity
0,2004-03-10 18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,2004-03-10 19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,2004-03-10 20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502


In [ ]:
required = [
    "RANDOM_STATE",
    "X_train",
    "y_train",
    "CV",
    "SCORING",
    "PRIMARY",
    "make_preprocessor",
    "run_search"
]

for name in required:
    print(name, ":", "Available" if name in globals() else "Missing")

RANDOM_STATE : Available
X_train : Missing
y_train : Missing
CV : Missing
SCORING : Missing
PRIMARY : Missing
make_preprocessor : Missing
run_search : Missing


In [ ]:

# ==========================================
# RESTORE SHARED SETUP FOR RANDOM FOREST
# ==========================================

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# 1. Random state
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 2. Clean the original dataset
df = df.replace(-200, np.nan)
df = df.drop(columns=["nmhc_gt"])
df = df.dropna(subset=["co_gt"]).reset_index(drop=True)

# 3. Create time features
df["hour"] = df["datetime"].dt.hour
df["weekday"] = df["datetime"].dt.weekday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# 4. Define features and target
SENSOR_COLS = [
    "pt08_s1_co", "pt08_s2_nmhc", "pt08_s3_nox",
    "pt08_s4_no2", "pt08_s5_o3"
]

WEATHER_COLS = [
    "temperature_c", "relative_humidity_pct",
    "absolute_humidity"
]

TIME_COLS = [
    "hour_sin", "hour_cos", "weekday", "is_weekend"
]

FEATURES = SENSOR_COLS + WEATHER_COLS + TIME_COLS
TARGET = "co_gt"

# 5. Chronological 80/20 split
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

# 6. Five-fold time-series cross-validation
CV = TimeSeriesSplit(n_splits=5)

# 7. Evaluation metrics
TOLERANCE = 0.5
TARGET_PCT = 90
LATENCY_LIMIT_MS = 1.0

def pct_within(y_true, y_pred, tol=TOLERANCE):
    return 100 * np.mean(
        np.abs(np.asarray(y_true) - np.asarray(y_pred)) <= tol
    )

SCORING = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
    "within": make_scorer(pct_within)
}

PRIMARY = "mae"

# 8. Preprocessing function
def make_preprocessor(scale=True):
    steps = [
        ("impute", SimpleImputer(strategy="median"))
    ]

    if scale:
        steps.append(("scale", StandardScaler()))

    return ColumnTransformer([
        ("num", Pipeline(steps), FEATURES)
    ])

# 9. Results evaluation
results = []

def evaluate_search(name, gs):
    r = gs.cv_results_
    i = gs.best_index_

    row = {
        "model": name,
        "train_MAE": -r["mean_train_mae"][i],
        "val_MAE": -r["mean_test_mae"][i],
        "val_MAE_std": r["std_test_mae"][i],
        "train_R2": r["mean_train_r2"][i],
        "val_R2": r["mean_test_r2"][i],
        "train_within_%": r["mean_train_within"][i],
        "val_within_%": r["mean_test_within"][i],
        "mean_fit_time_s": r["mean_fit_time"][i],
        "best_params": gs.best_params_
    }

    row["overfit_gap_MAE"] = (
        row["val_MAE"] - row["train_MAE"]
    )

    results.append(row)
    return pd.Series(row)

# 10. Grid search function
def run_search(name, pipe, grid):
    gs = GridSearchCV(
        pipe,
        param_grid=grid,
        cv=CV,
        scoring=SCORING,
        refit=PRIMARY,
        return_train_score=True,
        n_jobs=-1
    )

    gs.fit(X_train, y_train)

    return gs, evaluate_search(name, gs)

# 11. Confirm restored setup
print("Shared setup restored successfully!")
print("Cleaned dataset:", df.shape)
print("Features:", len(FEATURES))
print("Training data:", X_train.shape)
print("Test data:", X_test.shape)
print("Cross-validation folds:", CV.get_n_splits())
print("Random state:", RANDOM_STATE)


Shared setup restored successfully!
Cleaned dataset: (7674, 18)
Features: 12
Training data: (6139, 12)
Test data: (1535, 12)
Cross-validation folds: 5
Random state: 42


In [ ]:

# ==========================================
# RANDOM FOREST — REGULARISED EXPERIMENT
# ==========================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# Create Random Forest pipeline
rf_pipe = Pipeline([
    ("prep", make_preprocessor(scale=False)),
    ("model", RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=1
    ))
])

# Hyperparameter grid: 8 configurations
rf_grid_regularised = {
    "model__n_estimators": [300],
    "model__max_depth": [10, 20],
    "model__min_samples_leaf": [5, 10],
    "model__max_features": [0.7, "sqrt"]
}

print("Random Forest pipeline created.")
print("Configurations:", 8)
print("Cross-validation folds:", CV.get_n_splits())

# Train and evaluate
rf_gs_reg, rf_res_reg = run_search(
    "Random Forest (Regularised)",
    rf_pipe,
    rf_grid_regularised
)

print("\nRegularised Random Forest results:")
display(rf_res_reg)


Random Forest pipeline created.
Configurations: 8
Cross-validation folds: 5

Regularised Random Forest results:


,0
model,Random Forest (Regularised)
train_MAE,0.132556
val_MAE,0.370515
val_MAE_std,0.154021
train_R2,0.975935
val_R2,0.829105
train_within_%,96.790916
val_within_%,78.84653
mean_fit_time_s,7.984172
best_params,"{'model__max_depth': 20, 'model__max_features'..."


In [ ]:
print("Best regularised Random Forest parameters:")
print(rf_gs_reg.best_params_)

Best regularised Random Forest parameters:
{'model__max_depth': 20, 'model__max_features': 0.7, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import time
import numpy as np

# Final selected Random Forest
rf_final = Pipeline([
    ("prep", make_preprocessor(scale=False)),
    ("model", RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=3,
        max_features=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Train using training data only
rf_final.fit(X_train, y_train)

# Predict on untouched test data ONCE
start = time.perf_counter()
y_test_pred = rf_final.predict(X_test)
prediction_time = time.perf_counter() - start

# Calculate final test metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)
test_within = pct_within(y_test, y_test_pred)

latency_ms = prediction_time * 1000 / len(X_test)

print("FINAL RANDOM FOREST TEST RESULTS")
print("--------------------------------")
print("Test MAE:", round(test_mae, 4))
print("Test RMSE:", round(test_rmse, 4))
print("Test R2:", round(test_r2, 4))
print("Within tolerance (%):", round(test_within, 2))
print("Prediction latency (ms/sample):", round(latency_ms, 4))
print("Meets 90% tolerance target:", test_within >= TARGET_PCT)
print("Meets 1 ms latency limit:", latency_ms <= LATENCY_LIMIT_MS)


FINAL RANDOM FOREST TEST RESULTS
--------------------------------
Test MAE: 0.3687
Test RMSE: 0.6002
Test R2: 0.8007
Within tolerance (%): 77.52
Prediction latency (ms/sample): 0.1092
Meets 90% tolerance target: False
Meets 1 ms latency limit: True


In [ ]:
import pandas as pd

feature_importance = pd.Series(
    rf_final.named_steps["model"].feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print("Random Forest Feature Importance:")
display(feature_importance.to_frame("Importance"))

Random Forest Feature Importance:


,Importance
pt08_s2_nmhc,0.860390
temperature_c,0.029803
pt08_s1_co,0.021640
hour_sin,0.016323
pt08_s3_nox,0.015373
weekday,0.013687
hour_cos,0.010848
pt08_s5_o3,0.010754
absolute_humidity,0.007104
pt08_s4_no2,0.006184


In [ ]:
# Diagnose Random Forest prediction errors

errors = np.abs(np.asarray(y_test) - np.asarray(y_test_pred))

print("Total test predictions:", len(errors))
print("Predictions within tolerance:", np.sum(errors <= TOLERANCE))
print("Predictions outside tolerance:", np.sum(errors > TOLERANCE))
print("Average absolute error:", round(np.mean(errors), 4))
print("Median absolute error:", round(np.median(errors), 4))
print("90th percentile error:", round(np.percentile(errors, 90), 4))
print("Maximum error:", round(np.max(errors), 4))

Total test predictions: 1535
Predictions within tolerance: 1190
Predictions outside tolerance: 345
Average absolute error: 0.3687
Median absolute error: 0.2036
90th percentile error: 0.9584
Maximum error: 4.4116


In [ ]:
# Compare actual CO values with prediction errors

diagnostic_df = pd.DataFrame({
    "Actual_CO": np.asarray(y_test),
    "Predicted_CO": np.asarray(y_test_pred),
    "Absolute_Error": errors
})

diagnostic_df["Within_Tolerance"] = (
    diagnostic_df["Absolute_Error"] <= TOLERANCE
)

diagnostic_df.groupby(
    pd.cut(diagnostic_df["Actual_CO"], bins=5)
).agg(
    Count=("Absolute_Error", "count"),
    Mean_Error=("Absolute_Error", "mean"),
    Within_Tolerance=("Within_Tolerance", "mean")
)

/tmp/ipykernel_1600/130153051.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  diagnostic_df.groupby(


,Count,Mean_Error,Within_Tolerance
Actual_CO,,,
"(0.0917, 1.76]",822,0.250790,0.872263
"(1.76, 3.42]",500,0.381440,0.770000
"(3.42, 5.08]",151,0.688021,0.397351
"(5.08, 6.74]",54,0.950421,0.481481
"(6.74, 8.4]",8,1.733233,0.250000


In [ ]:
print("Training target distribution:")
print(y_train.describe())

print("\nTraining CO values above 3.42:")
print((y_train > 3.42).sum())

print("Training CO values above 5.08:")
print((y_train > 5.08).sum())

Training target distribution:
count    6139.000000
mean        2.190406
std         1.476797
min         0.100000
25%         1.100000
50%         1.900000
75%         2.900000
max        11.900000
Name: co_gt, dtype: float64

Training CO values above 3.42:
1059
Training CO values above 5.08:
282


In [ ]:
rf_grid_improved = {
    "model__n_estimators": [400, 600],
    "model__max_depth": [None, 20],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": [1.0]
}

rf_gs_improved, rf_res_improved = run_search(
    "Random Forest (Improved)",
    rf_pipe,
    rf_grid_improved
)

In [ ]:
print(rf_res_improved)

model                                       Random Forest (Improved)
train_MAE                                                   0.086779
val_MAE                                                     0.369074
val_MAE_std                                                 0.164222
train_R2                                                    0.988883
val_R2                                                      0.827271
train_within_%                                             98.862278
val_within_%                                               79.257087
mean_fit_time_s                                            29.866148
best_params        {'model__max_depth': None, 'model__max_feature...
overfit_gap_MAE                                             0.282295
dtype: object


In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import FunctionTransformer

rf_log_pipe = Pipeline([
    ("prep", make_preprocessor(scale=False)),
    ("model", TransformedTargetRegressor(
        regressor=RandomForestRegressor(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=3,
            max_features=1.0,
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

rf_log_grid = {
    "model__regressor__min_samples_leaf": [3, 5, 10],
    "model__regressor__max_depth": [None, 20]
}

rf_log_gs, rf_log_res = run_search(
    "Random Forest (Log Target)",
    rf_log_pipe,
    rf_log_grid
)

In [ ]:
print(rf_log_res)

model                                     Random Forest (Log Target)
train_MAE                                                   0.129277
val_MAE                                                     0.370938
val_MAE_std                                                 0.164596
train_R2                                                    0.976434
val_R2                                                      0.828759
train_within_%                                             96.919248
val_within_%                                               79.042033
mean_fit_time_s                                            14.690478
best_params        {'model__regressor__max_depth': None, 'model__...
overfit_gap_MAE                                             0.241661
dtype: object


In [ ]:
# Give higher CO concentrations more training weight

sample_weights = np.where(
    np.asarray(y_train) > 3.42,
    2.0,
    1.0
)

print("Normal-weight samples:", np.sum(sample_weights == 1.0))
print("Higher-weight samples:", np.sum(sample_weights == 2.0))

Normal-weight samples: 5080
Higher-weight samples: 1059


In [ ]:
rf_weighted = Pipeline([
    ("prep", make_preprocessor(scale=False)),
    ("model", RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=3,
        max_features=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_weighted.fit(
    X_train,
    y_train,
    model__sample_weight=sample_weights
)

print("Weighted Random Forest training completed!")

Weighted Random Forest training completed!


In [ ]:
from sklearn.model_selection import cross_validate

weighted_cv = cross_validate(
    rf_weighted,
    X_train,
    y_train,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=True
)

print("Weighted RF validation MAE:",
      round(-weighted_cv["test_mae"].mean(), 4))

print("Weighted RF validation within tolerance:",
      round(weighted_cv["test_within"].mean(), 2), "%")

Weighted RF validation MAE: 0.3679
Weighted RF validation within tolerance: 79.3 %


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

cv = TimeSeriesSplit(n_splits=5)

print("Cross-validation restored:", cv)

Cross-validation restored: TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)


In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_validate

weight_results = []

for weight in [1.5, 3.0, 5.0]:

    weights = np.where(
        np.asarray(y_train) > 3.42,
        weight,
        1.0
    )

    model = clone(rf_pipe)

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,
        params={"model__sample_weight": weights}
    )

    weight_results.append({
        "Weight": weight,
        "Validation_MAE": -scores["test_mae"].mean(),
        "Within_Tolerance_%": scores["test_within"].mean()
    })

weight_results_df = pd.DataFrame(weight_results)
display(weight_results_df.round(4))

,Weight,Validation_MAE,Within_Tolerance_%
0,1.5,0.3700,79.2571
1,3.0,0.3695,79.4526
2,5.0,0.3704,79.1984


In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_validate

range_weight_results = []

for high_weight in [3.0, 5.0, 8.0]:

    weights = np.ones(len(y_train))

    weights[np.asarray(y_train) > 3.42] = 3.0
    weights[np.asarray(y_train) > 5.08] = high_weight

    model = clone(rf_pipe)

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,
        params={"model__sample_weight": weights}
    )

    range_weight_results.append({
        "High_CO_Weight": high_weight,
        "Validation_MAE": -scores["test_mae"].mean(),
        "Within_Tolerance_%": scores["test_within"].mean()
    })

display(pd.DataFrame(range_weight_results).round(4))

,High_CO_Weight,Validation_MAE,Within_Tolerance_%
0,3.0,0.3695,79.4526
1,5.0,0.3718,79.4330
2,8.0,0.3714,79.4135


In [ ]:
y_train = train_df["co_gt"]

## Part C — M3: HistGradientBoosting


## Part E — M1: overfitting analysis (learning curves, per-fold scores over time)


## Part F — M5: computational cost (n_jobs=1 timing, latency vs C2, scalability)


## Part G — M5: final recommended pipeline, scored once on the test set
